In [ ]:
pip install "scgraph>=4.0.0"

In [ ]:
pip install folium==0.16.0

In [ ]:
from scgraph import GeoGraph

In [ ]:
world_sea = GeoGraph.load_geograph("world_sea")

# Get the shortest path between shanghai and savannah
sea_output = world_sea.get_shortest_path(
    origin_node={"latitude": 31.23,"longitude": 121.47}, 
    destination_node={"latitude": 32.08,"longitude": -81.09}
)
print('Length: ',sea_output['length'])
print('Path: ', sea_output['coordinate_path'])

In [ ]:
# A released network of a part of the world holds its roads, its railways, and the sea lanes and flights of the world
oceania = GeoGraph.load_geograph("oceania")

# Get the shortest path between Sydney and Perth
oceania_output = oceania.get_shortest_path(
    origin_node={"latitude": -33.87,"longitude": 151.21}, 
    destination_node={"latitude": -31.95,"longitude": 115.86}
)
print('Length: ',oceania_output['length'])


In [ ]:
# A simple example of how to use folium to visualize the shortest path between two points
import folium
world_sea = GeoGraph.load_geograph("world_sea")

# Get the shortest path between shanghai and houston
folium_output = world_sea.get_shortest_path(
    origin_node={"latitude": 31.23,"longitude": 121.47}, 
    destination_node={"latitude": 29.76,"longitude": -95.37}
)

# Folium does not wrap coordinates around the globe, so we need to adjust the path accordingly
# Essentially we create 5 differnt paths that offset the original path by 0, 360, -360, 720, -720 degrees
# This allows the illusion that the path wraps around the globe with folium
def adjustArcPath(path):
    for index in range(1, len(path)):
        x = path[index][1]
        prevX = path[index - 1][1]
        path[index][1] = x - (round((x - prevX)/360,0) * 360)
    return path

def modifyArcPathLong(points, amount):
    return [[i[0], i[1]+amount] for i in points]

def getCleanArcPath(path):
    path = adjustArcPath(path)
    return [
        path,
        modifyArcPathLong(path, 360),
        modifyArcPathLong(path, -360),
        modifyArcPathLong(path, 720),
        modifyArcPathLong(path, -720)
    ]

# Create a folium map
map = folium.Map([40, 180], zoom_start=2)
# Populate it with the path
folium.PolyLine(
    getCleanArcPath(folium_output['coordinate_path']),
    color='green',
    weight=5,
    opacity=0.5,
    popup='Length (KM): ' + str(folium_output['length'])
).add_to(map)
map